# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
from tqdm.notebook import tqdm
from itertools import product
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, VotingClassifier,
    BaggingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df['dayofweek'] = pd.read_csv('../data/dayofweek.csv')['dayofweek']

print(df.shape)
df.head()

(1686, 44)


,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [3]:
X = df.drop(columns=['dayofweek'])
y = df['dayofweek']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=21, stratify=y_train_full
)

print(f"X_train: {X_train.shape}, X_valid: {X_valid.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_valid: {y_valid.shape}, y_test: {y_test.shape}")

X_train: (1078, 43), X_valid: (270, 43), X_test: (338, 43)
y_train: (1078,), y_valid: (270,), y_test: (338,)


## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [4]:
svm = SVC(
    C=10, gamma="auto", kernel="rbf", probability=True, random_state=21
    ).fit(X_train, y_train)
y_pred_svm = svm.predict(X_valid)

print(f"accuracy is {accuracy_score(y_valid, y_pred_svm):.5f}")
print(f"precision is {precision_score(y_valid, y_pred_svm, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred_svm, average='weighted'):.5f}")

accuracy is 0.87778
precision is 0.88162
recall is 0.87778


In [5]:
tree = DecisionTreeClassifier(
    class_weight="balanced", criterion="gini", max_depth=22, random_state=21
    ).fit(X_train, y_train)
y_pred_tree = tree.predict(X_valid)

print(f"accuracy is {accuracy_score(y_valid, y_pred_tree):.5f}")
print(f"precision is {precision_score(y_valid, y_pred_tree, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred_tree, average='weighted'):.5f}")

accuracy is 0.86667
precision is 0.86984
recall is 0.86667


In [6]:
rf = RandomForestClassifier(
    n_estimators=50, max_depth=28, class_weight=None, criterion="gini", random_state=21
    ).fit(X_train, y_train)
y_pred_rf = rf.predict(X_valid)

print(f"accuracy is {accuracy_score(y_valid, y_pred_rf):.5f}")
print(f"precision is {precision_score(y_valid, y_pred_rf, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred_rf, average='weighted'):.5f}")

accuracy is 0.89259
precision is 0.89361
recall is 0.89259


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [7]:
voting_hard = VotingClassifier(
    estimators=[("svm", svm), ("tree", tree), ("rf", rf)], voting="hard"
    ).fit(X_train, y_train)
y_pred_vh = voting_hard.predict(X_valid)

print("Hard Voting:")
print(f"accuracy is {accuracy_score(y_valid, y_pred_vh):.5f}")
print(f"precision is {precision_score(y_valid, y_pred_vh, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred_vh, average='weighted'):.5f}")

Hard Voting:
accuracy is 0.89630
precision is 0.89605
recall is 0.89630


In [8]:
voting_soft = VotingClassifier(
    estimators=[("svm", svm), ("tree", tree), ("rf", rf)], voting="soft"
    ).fit(X_train, y_train)
y_pred_vs = voting_soft.predict(X_valid)

print("Soft Voting:")
print(f"accuracy is {accuracy_score(y_valid, y_pred_vs):.5f}")
print(f"precision is {precision_score(y_valid, y_pred_vs, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred_vs, average='weighted'):.5f}")

Soft Voting:
accuracy is 0.88148
precision is 0.88418
recall is 0.88148


In [9]:
weight_combos = [w for w in product(range(1, 5), repeat=3) if any(w)]

results = []
for w in tqdm(weight_combos, desc="Soft voting (веса)"):
    vc = VotingClassifier(
        estimators=[("svm", svm), ("tree", tree), ("rf", rf)],
        voting="soft", weights=w
        ).fit(X_train, y_train)
    y_pred = vc.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average="weighted")
    rec = recall_score(y_valid, y_pred, average="weighted")
    results.append({"weights": w, "accuracy": acc, "precision": prec, "recall": rec})

results_df = pd.DataFrame(results).sort_values(
    ["accuracy", "precision"], ascending=False
)
results_df


Soft voting (веса):   0%|          | 0/64 [00:00<?, ?it/s]

,weights,accuracy,precision,recall
48,"(4, 1, 1)",0.907407,0.910258,0.907407
50,"(4, 1, 3)",0.907407,0.909871,0.907407
49,"(4, 1, 2)",0.907407,0.909683,0.907407
32,"(3, 1, 1)",0.903704,0.905991,0.903704
51,"(4, 1, 4)",0.903704,0.904985,0.903704
...,...,...,...,...
29,"(2, 4, 2)",0.866667,0.869843,0.866667
40,"(3, 3, 1)",0.866667,0.869843,0.866667
44,"(3, 4, 1)",0.866667,0.869843,0.866667
45,"(3, 4, 2)",0.866667,0.869843,0.866667


In [10]:
results_hard = []
for w in tqdm(weight_combos, desc="Hard voting (веса)"):
    vc = VotingClassifier(
        estimators=[("svm", svm), ("tree", tree), ("rf", rf)],
        voting="hard", weights=w
        ).fit(X_train, y_train)
    y_pred = vc.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average="weighted")
    rec = recall_score(y_valid, y_pred, average="weighted")
    results_hard.append({"weights": w, "voting": "hard", "accuracy": acc, "precision": prec, "recall": rec})

results_hard_df = pd.DataFrame(results_hard).sort_values(
    ["accuracy", "precision"], ascending=False
)
results_hard_df

Hard voting (веса):   0%|          | 0/64 [00:00<?, ?it/s]

,weights,voting,accuracy,precision,recall
25,"(2, 3, 2)",hard,0.907407,0.908272,0.907407
30,"(2, 4, 3)",hard,0.907407,0.908272,0.907407
45,"(3, 4, 2)",hard,0.907407,0.908272,0.907407
46,"(3, 4, 3)",hard,0.907407,0.908272,0.907407
5,"(1, 2, 2)",hard,0.903704,0.904407,0.903704
...,...,...,...,...,...
52,"(4, 2, 1)",hard,0.877778,0.881615,0.877778
8,"(1, 3, 1)",hard,0.866667,0.869843,0.866667
12,"(1, 4, 1)",hard,0.866667,0.869843,0.866667
13,"(1, 4, 2)",hard,0.866667,0.869843,0.866667


In [11]:
results_df['voting'] = 'soft'
all_voting = pd.concat([results_df, results_hard_df], ignore_index=True).sort_values(
    ['accuracy', 'precision'], ascending=False
    ).reset_index(drop=True)

best_row = all_voting.iloc[0]
best_weights = best_row['weights']
best_voting_type = best_row['voting']
print(f'Лучший тип голосования: {best_voting_type}, weights={best_weights}')
print(f' Accuracy: {best_row["accuracy"]:.5f}')

best_voting = VotingClassifier(
    estimators=[("svm", svm), ("tree", tree), ("rf", rf)],
    voting=best_voting_type, weights=best_weights
    ).fit(X_train, y_train)

Лучший тип голосования: soft, weights=(4, 1, 1)
 Accuracy: 0.90741


In [12]:
y_pred_test_voting = best_voting.predict(X_test)

print(f"Лучший классификатор голосования на тестовом датасете ({best_voting_type}, weights={best_weights}):")
print(f"accuracy is {accuracy_score(y_test, y_pred_test_voting):.5f}")
print(f"precision is {precision_score(y_test, y_pred_test_voting, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred_test_voting, average='weighted'):.5f}")

Лучший классификатор голосования на тестовом датасете (soft, weights=(4, 1, 1)):
accuracy is 0.89941
precision is 0.90357
recall is 0.89941


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [13]:
base_svm = SVC(C=10, gamma='auto', kernel='rbf', probability=True, random_state=21)

n_estimators_list = np.arange(5, 55, 5)
bag_results = []

for n in tqdm(n_estimators_list, desc='Bagging n_estimators'):
    bag = BaggingClassifier(
        estimator=base_svm, n_estimators=n, random_state=21
        ).fit(X_train, y_train)
    y_pred = bag.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    rec = recall_score(y_valid, y_pred, average='weighted')
    bag_results.append({'n_estimators': n, 'accuracy': acc, 'precision': prec, 'recall': rec})

bag_df = pd.DataFrame(bag_results).sort_values(['accuracy', 'precision'], ascending=False)
bag_df

Bagging n_estimators:   0%|          | 0/10 [00:00<?, ?it/s]

,n_estimators,accuracy,precision,recall
5,30,0.888889,0.897182,0.888889
4,25,0.888889,0.896169,0.888889
1,10,0.885185,0.894269,0.885185
2,15,0.885185,0.892994,0.885185
3,20,0.885185,0.892576,0.885185
7,40,0.881481,0.891108,0.881481
8,45,0.881481,0.890776,0.881481
9,50,0.881481,0.890354,0.881481
6,35,0.877778,0.887873,0.877778
0,5,0.874074,0.884229,0.874074


In [14]:
param_grid = list(product(
    np.arange(5, 55, 5), [0.5, 0.7, 0.8, 1.0], [0.5, 0.7, 0.8, 1.0],
    [True, False]
))

bag_grid_results = []
for n, ms, mf, bf in tqdm(param_grid, desc='Bagging (параметры)'):
    bag = BaggingClassifier(
        estimator=base_svm, n_estimators=int(n),
        max_samples=ms, max_features=mf,
        bootstrap_features=bf, random_state=21,
    ).fit(X_train, y_train)
    y_pred = bag.predict(X_valid)
    bag_grid_results.append({
        'n_estimators': int(n), 'max_samples': ms, 'max_features': mf,
        'bootstrap_features': bf,
        'accuracy': accuracy_score(y_valid, y_pred),
        'precision': precision_score(y_valid, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_valid, y_pred, average='weighted')
    })

bag_grid_df = pd.DataFrame(bag_grid_results).sort_values(
    ['accuracy', 'precision'], ascending=False
).reset_index(drop=True)
bag_grid_df.head(10)

Bagging (параметры):   0%|          | 0/320 [00:00<?, ?it/s]

,n_estimators,max_samples,max_features,bootstrap_features,accuracy,precision,recall
0,30,1.0,1.0,False,0.888889,0.897182,0.888889
1,25,1.0,1.0,False,0.888889,0.896169,0.888889
2,10,1.0,1.0,False,0.885185,0.894269,0.885185
3,15,1.0,1.0,False,0.885185,0.892994,0.885185
4,20,1.0,1.0,False,0.885185,0.892576,0.885185
5,40,1.0,1.0,False,0.881481,0.891108,0.881481
6,45,1.0,1.0,False,0.881481,0.890776,0.881481
7,50,1.0,1.0,False,0.881481,0.890354,0.881481
8,35,1.0,1.0,False,0.877778,0.887873,0.877778
9,40,1.0,0.8,False,0.877778,0.887078,0.877778


In [15]:
best_bag_row = bag_grid_df.iloc[0]
best_bagging = BaggingClassifier(
    estimator=base_svm,
    n_estimators=int(best_bag_row['n_estimators']),
    max_samples=best_bag_row['max_samples'], max_features=best_bag_row['max_features'],
    bootstrap_features=bool(best_bag_row['bootstrap_features']), random_state=21
).fit(X_train, y_train)

y_pred_test_bag = best_bagging.predict(X_test)

print(f"Лучший классификатор бэггинга на тестовом датасете "
      f"(n={int(best_bag_row['n_estimators'])}, ms={best_bag_row['max_samples']}, "
      f"mf={best_bag_row['max_features']}, bf={bool(best_bag_row['bootstrap_features'])}):")
print(f"accuracy is {accuracy_score(y_test, y_pred_test_bag):.5f}")
print(f"precision is {precision_score(y_test, y_pred_test_bag, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred_test_bag, average='weighted'):.5f}")

Лучший классификатор бэггинга на тестовом датасете (n=30, ms=1.0, mf=1.0, bf=False):
accuracy is 0.87278
precision is 0.87840
recall is 0.87278


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [16]:
n_splits_list = [2, 3, 4, 5, 6, 7]
passthrough_list = [True, False]

stack_results = []
for n_splits in tqdm(n_splits_list, desc="Stacking n_splits"):
    for pt in passthrough_list:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        stacking = StackingClassifier(
            estimators=[("svm", svm), ("tree", tree), ("rf", rf)],
            final_estimator=OneVsRestClassifier(LogisticRegression(solver="liblinear")),
            cv=cv, passthrough=pt
            ).fit(X_train, y_train)
        y_pred = stacking.predict(X_valid)
        acc = accuracy_score(y_valid, y_pred)
        prec = precision_score(y_valid, y_pred, average="weighted")
        rec = recall_score(y_valid, y_pred, average="weighted")
        stack_results.append({
            "n_splits": n_splits, "passthrough": pt,
            "accuracy": acc, "precision": prec, "recall": rec
        })

stack_df = pd.DataFrame(stack_results).sort_values(
    ["accuracy", "precision"], ascending=False
)
stack_df

Stacking n_splits:   0%|          | 0/6 [00:00<?, ?it/s]

,n_splits,passthrough,accuracy,precision,recall
7,5,False,0.914815,0.916587,0.914815
6,5,True,0.911111,0.914523,0.911111
5,4,False,0.907407,0.909423,0.907407
10,7,True,0.907407,0.909015,0.907407
8,6,True,0.907407,0.908809,0.907407
4,4,True,0.903704,0.906421,0.903704
11,7,False,0.903704,0.904871,0.903704
9,6,False,0.903704,0.904820,0.903704
0,2,True,0.903704,0.904603,0.903704
2,3,True,0.900000,0.901721,0.900000


In [17]:
best_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)
best_stacking = StackingClassifier(
    estimators=[("svm", svm), ("tree", tree), ("rf", rf)],
    final_estimator=OneVsRestClassifier(LogisticRegression(solver="liblinear")),
    cv=best_cv, passthrough=False
    ).fit(X_train, y_train)

In [18]:
y_pred_test_stack = best_stacking.predict(X_test)

print(f"Лучший стекинг-классификатор на тестовом датасете (n_splits=5, passthrough=False):")
print(f"accuracy is {accuracy_score(y_test, y_pred_test_stack):.5f}")
print(f"precision is {precision_score(y_test, y_pred_test_stack, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred_test_stack, average='weighted'):.5f}")

Лучший стекинг-классификатор на тестовом датасете (n_splits=5, passthrough=False):
accuracy is 0.90828
precision is 0.91189
recall is 0.90828


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [19]:
models_test = {
    "VotingClassifier": {
        "model": best_voting, "y_pred": y_pred_test_voting,
        "accuracy": accuracy_score(y_test, y_pred_test_voting),
        "precision": precision_score(y_test, y_pred_test_voting, average="weighted")
    },
    "BaggingClassifier": {
        "model": best_bagging, "y_pred": y_pred_test_bag,
        "accuracy": accuracy_score(y_test, y_pred_test_bag),
        "precision": precision_score(y_test, y_pred_test_bag, average="weighted")
    },
    "StackingClassifier": {
        "model": best_stacking, "y_pred": y_pred_test_stack,
        "accuracy": accuracy_score(y_test, y_pred_test_stack),
        "precision": precision_score(y_test, y_pred_test_stack, average="weighted")
    }
}

best_name = max(models_test, key=lambda k: (models_test[k]["accuracy"], models_test[k]["precision"]))
best_model = models_test[best_name]["model"]
best_y_pred = models_test[best_name]["y_pred"]

print(f"Лучшая модель ансамбля: {best_name}")
print(f"accuracy is {models_test[best_name]['accuracy']:.5f}")
print(f"precision is {models_test[best_name]['precision']:.5f}")
print(f"recall is {recall_score(y_test, best_y_pred, average='weighted'):.5f}")

Лучшая модель ансамбля: StackingClassifier
accuracy is 0.90828
precision is 0.91189
recall is 0.90828


In [20]:
days = {0: 'Понедельник', 1: 'Вторник', 2: 'Среда', 3: 'Четверг',
       4: 'Пятница', 5: 'Суббота', 6: 'Воскресенье'}

df_full = df.copy()
df_full['labname'] = df_full.filter(like='labname_').idxmax(axis=1).str.replace('labname_', '')
df_full['uid'] = df_full.filter(like='uid_').idxmax(axis=1).str.replace('uid_', '')

df_test = df_full.loc[X_test.index].copy()
df_test['y_pred'] = best_y_pred
df_test['error'] = (df_test['dayofweek'] != df_test['y_pred']).astype(int)

totals = df_full.groupby('dayofweek').size()
errors = df_test.groupby('dayofweek')['error'].sum()
day_pct = (errors / totals * 100).sort_index()

print("Ошибки по дням недели (% от числа образцов класса в полном датасете)")
for d, pct in day_pct.items():
    print(f"  {days[d]:15s}- {errors[d]:3.0f} ошибок из {totals[d]:3d} значений = {pct:.1f}%")
print(f"\nБольше всего ошибок: {days[day_pct.idxmax()]} ({day_pct.max():.1f}%)")

lab_pct = (df_test.groupby('labname')['error'].sum() / df_full.groupby('labname').size() * 100).sort_values(ascending=False)
print("\nОшибки по labname (топ-5)")
print(lab_pct.rename_axis(None).head(5).to_string())
print(f"\nБольше всего ошибок: {lab_pct.idxmax()} ({lab_pct.max():.1f}%)")

user_pct = (df_test.groupby('uid')['error'].sum() / df_full.groupby('uid').size() * 100).sort_values(ascending=False)
print("\nОшибки по пользователям (топ-5)")
print(user_pct.rename_axis(None).head(5).to_string())
print(f"\nБольше всего ошибок: {user_pct.idxmax()} ({user_pct.max():.1f}%)")


Ошибки по дням недели (% от числа образцов класса в полном датасете)
  Понедельник    -   7 ошибок из 136 значений = 5.1%
  Вторник        -   3 ошибок из 274 значений = 1.1%
  Среда          -   3 ошибок из 149 значений = 2.0%
  Четверг        -   3 ошибок из 396 значений = 0.8%
  Пятница        -   2 ошибок из 104 значений = 1.9%
  Суббота        -   7 ошибок из 271 значений = 2.6%
  Воскресенье    -   6 ошибок из 356 значений = 1.7%

Больше всего ошибок: Понедельник (5.1%)

Ошибки по labname (топ-5)
lab03      100.000000
laba04s      5.769231
laba04       4.494382
laba06s      3.278689
lab05s       2.777778

Больше всего ошибок: lab03 (100.0%)

Ошибки по пользователям (топ-5)
user_6     16.666667
user_22    14.285714
user_27     4.347826
user_2      3.305785
user_19     3.296703

Больше всего ошибок: user_6 (16.7%)


In [21]:
joblib.dump(best_model, 'ensemble_best_model.sav')

['ensemble_best_model.sav']